In [18]:
import cv2
import pytesseract
import pandas as pd
import numpy as np
import re

In [19]:
img = cv2.imread("invoices/batch1-0371.jpg")

In [20]:
# for reducing complexity greyscaling is implemented
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [21]:
text = pytesseract.image_to_string(gray)
print(text)

Invoice no: 39652805

Date of issue: 03/20/2020

Seller: Client:

Lewis and Sons Hancock LLC

256 Cruz Views Suite 230 04712 Andrew Meadows
East Justin, WA 50904 North Hannah, NV 36457
Tax Id: 988-97-7797 Tax Id: 982-72-2795

IBAN: GB51EXRQ57779404817003

ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross
worth
tks computer 1,00 each 299,99 299,99 10% 329,99
SUMMARY
VAT [%] Net worth VAT Gross worth
10% 299,99 30,00 329,99

Total $ 299,99 $ 30,00 $ 329,99



In [22]:
invoice_no = re.search(r"Invoice no:\s*(\d+)", text)
invoice_no = invoice_no.group(1) if invoice_no else ""

In [23]:
invoice_no

'39652805'

In [24]:
date = re.search(r"Date of issue:\s*([0-9/]+)", text)
date = date.group(1) if date else ""

In [25]:
date

'03/20/2020'

In [26]:
seller = re.search(r"Seller:\n([\s\S]*?)\n\n", text)
seller = seller.group(1).strip() if seller else ""

In [27]:
seller

''

In [28]:
client = re.search(r"Client:\n([\s\S]*?)\n\n", text)
client = client.group(1).strip() if client else ""

In [29]:
client

'Lewis and Sons Hancock LLC'

In [30]:
seller_tax = re.search(r"Tax Id:\s*([0-9\-]+)", text)
seller_tax = seller_tax.group(1) if seller_tax else ""

iban = re.search(r"IBAN:\s*([A-Z0-9]+)", text)
iban = iban.group(1) if iban else ""

client_tax = re.findall(r"Tax Id:\s*([0-9\-]+)", text)
client_tax = client_tax[1] if len(client_tax) > 1 else ""

seller_tax, iban, client_tax

('988-97-7797', 'GB51EXRQ57779404817003', '982-72-2795')

In [31]:
item_pattern = r"""
\d+\.\s+                # item number
([A-Za-z ]+)\s+         # description
([\d.,]+)\s+            # quantity
(\w+)\s+                # unit
([\d.,]+)\s+            # net price
([\d.,]+)\s+            # net worth
(\d+\s*%)\s+            # VAT
([\d.,]+)               # gross worth
"""

item = re.search(item_pattern, text, re.VERBOSE)


In [32]:
if item:
    item_data = {
        "description": item.group(1).strip(),
        "qty": item.group(2),
        "unit": item.group(3),
        "net_price": item.group(4),
        "net_worth": item.group(5),
        "vat": item.group(6),
        "gross_worth": item.group(7)
    }
    print(item_data)
else:
    print("Item not detected — check OCR output")


Item not detected — check OCR output


In [33]:
print(text[text.find("ITEMS"):])

ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross
worth
tks computer 1,00 each 299,99 299,99 10% 329,99
SUMMARY
VAT [%] Net worth VAT Gross worth
10% 299,99 30,00 329,99

Total $ 299,99 $ 30,00 $ 329,99



In [34]:
import os
import glob

def extract_invoice_data(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load image: {image_path}")
        return {}
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray)
    
    invoice_no = re.search(r"Invoice no:\s*(\d+)", text)
    invoice_no = invoice_no.group(1) if invoice_no else ""
    
    date = re.search(r"Date of issue:\s*([0-9/]+)", text)
    date = date.group(1) if date else ""
    
    seller = re.search(r"Seller:\n([\s\S]*?)\n\n", text)
    seller = seller.group(1).strip() if seller else ""
    
    client = re.search(r"Client:\n([\s\S]*?)\n\n", text)
    client = client.group(1).strip() if client else ""
    
    seller_tax = re.search(r"Tax Id:\s*([0-9\-]+)", text)
    seller_tax = seller_tax.group(1) if seller_tax else ""
    
    iban = re.search(r"IBAN:\s*([A-Z0-9]+)", text)
    iban = iban.group(1) if iban else ""
    
    client_tax = re.findall(r"Tax Id:\s*([0-9\-]+)", text)
    client_tax = client_tax[1] if len(client_tax) > 1 else ""
    
    return {
        "invoice_no": invoice_no,
        "date": date,
        "seller": seller,
        "client": client,
        "seller_tax": seller_tax,
        "iban": iban,
        "client_tax": client_tax
    }

invoice_folder = "invoices"
image_files = glob.glob(os.path.join(invoice_folder, "*.jpg"))
data = []
for image_file in image_files:
    data.append(extract_invoice_data(image_file))

df = pd.DataFrame(data)
df.to_csv("extracted_invoices.csv", index=False)
print("Data extracted and saved to extracted_invoices.csv")

Data extracted and saved to extracted_invoices.csv
